# msg_parts

> The canonical LLM message model - `Msg` and the `Part` classes - and its formatted text form

In [ ]:
#| default_exp msg_parts

`Msg` and the `Part` subclasses describe LLM conversations without tying them to a provider. `fastllm` uses these types for its chat clients. Dialog libraries, transcript tools, and compaction code can use them without importing an LLM client. This module depends on `fastcore`.

Use `mk_msg` and `mk_msgs` to build messages from ordinary Python values. Use `fmt2hist` and `hist2fmt` to convert between messages and editable Markdown replies. The Markdown format stores tool calls, results, and usage in fenced JSON blocks.

In [ ]:
#| export
import base64, json, copy
from json import dumps
from fastcore.utils import *
from fastcore.xtras import detect_mime

In [ ]:
#| hide
from fastcore.test import *
from fastcore.xml import Safe
from IPython.display import Markdown

## Part

A message can contain text, images, reasoning, and tool calls. Each item in `Msg.content` is a `Part` subclass. Provider converters translate between these objects and their API's content blocks. Changing providers doesn't require a different way to construct messages, although each provider has its own capability limits.

Subclasses register a serialization tag such as `text` or `input_image`. The tag is available as `.type`. Each subclass defines its own fields and display methods. Renderers call those methods without checking the tag.

Every part accepts `raw` and `cache_control`. `raw` retains provider details that the shared fields don't represent, such as an Anthropic thinking signature. `cache_control` holds a prompt-cache directive for providers that support one. `fastllm` uses these parts in input coercion, response normalization, and provider serialization.

The provider APIs use different names and nesting for the same content:

| API | Text | Image | File |
|---|---|---|---|
| [OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create) | `{"type":"input_text","text":"..."}` | `{"type":"input_image","image_url":"..."}` | `{"type":"input_file","file_data":"data:application/pdf;base64,...","filename":"..."}` |
| [OpenAI Chat](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create) | `{"type":"text","text":"..."}` | `{"type":"image_url","image_url":{"url":"..."}}` | `{"type":"file","file":{"file_data":"data:application/pdf;base64,...","filename":"..."}}` |
| [Anthropic](https://docs.anthropic.com/en/api/messages) | `{"type":"text","text":"..."}` | `{"type":"image","source":{"type":"base64","media_type":"image/jpeg","data":"..."}}` | `{"type":"document","source":{"type":"base64","media_type":"application/pdf","data":"..."}}` |
| [Gemini](https://ai.google.dev/api/generate-content) | `{"text":"..."}` | `{"inlineData":{"mimeType":"image/jpeg","data":"..."}}` | `{"fileData":{"mimeType":"application/pdf","fileUri":"..."}}` |

These are API representations, not constructor arguments for `Part`. Construct `Text`, `InputImage`, `InputAudio`, `InputVideo`, or `InputFile` instead. Media subclasses store a URL or data URL in `text` and a MIME type in `mime`. There is no `Part.data` field or provider-name override. `mk_part` requires the registered tag, not aliases such as `image_url` or `pdf`.

| Class | Registered tag | Content |
|---|---|---|
| `Text` | `text` | Plain text |
| `InputImage` | `input_image` | Image URL or data URL |
| `InputAudio` | `input_audio` | Audio URL or data URL |
| `InputVideo` | `input_video` | Video URL or data URL |
| `InputFile` | `input_file` | File URL or data URL |

The following table describes the current `fastllm` user-message converters. It doesn't promise that every model accepts every format its API can represent.

| Part | OpenAI Responses | OpenAI Chat | Anthropic | Gemini |
|---|---|---|---|---|
| `Text` | `input_text` | `text` | `text` | `text` |
| `InputImage` | `input_image` | `image_url` | `image` with `source` | `inlineData` or `fileData` |
| `InputAudio` | Raises `ValueError` | `input_audio` | Raises `ValueError` | `inlineData` or `fileData` |
| `InputVideo` | `input_video` with `video_url` | Raises `ValueError` | Raises `ValueError` | `inlineData` or `fileData` |
| `InputFile` | `input_file` | `file` | `document` with `source` | `inlineData` or `fileData` |

OpenAI Chat audio requires a base64 data URL. Its converter extracts the encoded data and a `wav` or `mp3` format into `input_audio`. The Chat file converter also requires a base64 data URL. It doesn't accept a URL or a file ID through `InputFile`. Responses accepts file URLs and data URLs. Anthropic uses a `url` or `base64` source. Gemini uses `fileData` for URLs and `inlineData` for data URLs.

The original API comparison used these specification locations: OpenAI Responses `Input{Text,Image,File}Content` (`specs/openai.with-code-samples.yml:68093-68444`), OpenAI Chat `ChatCompletionRequestMessageContentPart*` (`specs/openai.with-code-samples.yml:35185-35321`), Anthropic `Request{Text,Image,Document}Block` (`specs/anthropic.yml:12159-12458`), and Gemini `Part`/`Blob`/`FileData` (`specs/gemini.json:189-387`). For converter behavior, see `fastllm.openai_responses`, `fastllm.openai_chat`, `fastllm.anthropic`, and `fastllm.gemini`.

In [ ]:
#| export
class Part(BasicRepr):
    "Base class for content parts; subclasses register under the wire tag they serialize as."
    text = None    # subclasses that carry text declare it as an attribute; the rest read as None
    reg = {}       # wire tag -> subclass
    def __init__(self,
        *,
        raw=None,          # The vendor's original block, kept for lossless round-trips
        cache_control=None # Prompt-cache directive for providers that take one
    ):
        store_attr()
    def __init_subclass__(cls, tag=None, **kw):
        super().__init_subclass__(**kw)
        if tag: cls.type,Part.reg[tag] = tag,cls
    def __eq__(self, o): return type(o) is type(self) and self.__dict__ == o.__dict__
    def __hash__(self): return hash((type(self).__name__, self.text))
    def replace(self, **kw):
        "A copy of this part with `kw` attributes changed"
        res = copy.copy(self)
        res.__dict__.update(kw)
        return res

In [ ]:
#| export
PartType = str_enum('PartType', 'text', 'thinking', 'refusal', 'tool_use', 'tool_result',
    'input_image', 'input_audio', 'input_video', 'input_file')

`PartType` lists the registered tags. `ToolUse` and `ToolResult` are parts too. Their definitions appear in the tool section below.

In [ ]:
#| export
class Text(Part, tag=PartType.text):
    "Plain text content; `citations` lists the sources it rests on as `url_citation` dicts (`url`, `title`, optional `start_index`/`end_index`)"
    def __init__(self, text=None, citations=None, **kw):
        super().__init__(**kw)
        store_attr('text,citations')

class Thinking(Part, tag=PartType.thinking):
    "A reasoning block; `showthink` asks renderers to show the thought rather than a 🧠 glyph."
    def __init__(self, text=None, showthink=False, **kw):
        super().__init__(**kw)
        store_attr('text,showthink')

class Refusal(Part, tag=PartType.refusal):
    "A provider's refusal to answer."
    def __init__(self, text=None, **kw):
        super().__init__(**kw)
        store_attr('text')

`Media` stores the URL or data URL in `text` and its MIME type in `mime`.

In [ ]:
#| export
class Media(Part):
    "Media content: `text` is a URL or data URL."
    def __init__(self, text=None, mime=None, **kw):
        super().__init__(**kw)
        store_attr('text,mime')

Choose the subclass for the media you want to send:

In [ ]:
#| export
# chkstyle: skip
class InputImage(Media, tag=PartType.input_image): "An image input."
class InputAudio(Media, tag=PartType.input_audio): "An audio input."
class InputVideo(Media, tag=PartType.input_video): "A video input."
class InputFile (Media, tag=PartType.input_file ): "A file input."

Use `mk_part` when reading serialized parts. It looks up the tag in `Part.reg` and passes the remaining fields to that class.

In [ ]:
#| export
def mk_part(type, **kw):
    "The `Part` subclass registered for wire tag `type`, built from `kw`"
    return Part.reg[type](**kw)

In [ ]:
#| export
def _trunc_strs(o, n=200):
    "Truncate str or dict"
    if not o: return o
    if isinstance(o,str) and len(o)>n: return o[:100]+'...'
    if isinstance(o,dict): return {k: (v[:100]+'...' if isinstance(v,str) and len(v)>n else v) for k,v in o.items()}
    return o

@patch
def _repr_markdown_(self:Part):
    flds = '\n'.join(f"- {k}: `{_trunc_strs(getattr(self,k))}`" for k in self.__dict__ if k[0] != '_' and k not in ('text','cache_control'))
    return f"""**{type(self).__name__}** (`{self.type}`)

{_trunc_strs(self.text) if self.text else ''}

::: details

{flds}

:::"""

In [ ]:
Text('Hello world!'*30, raw={'long':"10"*150})

**Text** (`text`)

Hello world!Hello world!Hello world!Hello world!Hello world!Hello world!Hello world!Hello world!Hell...

::: details

- raw: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`
- citations: `None`

:::

## Msg

`Msg` groups an ordered list of parts under a role such as `user`, `assistant`, or `tool`. Its `text` property joins the `Text` parts without separators. Other parts remain in `content`.

`fastllm` accepts these messages in completion requests and uses them to replay tool conversations. A `Completion` contains the assistant's `Msg`. Provider converters handle the differences in message roles and layout.

OpenAI Chat uses `{role, content}` messages. Its roles include `system`, `developer`, `user`, `assistant`, and `tool`. Content can be a string or typed blocks. Responses accepts message input items but represents tool calls and results as separate items.

Anthropic accepts `user` and `assistant` messages. It takes the system prompt in a separate `system` parameter. Gemini uses `{role, parts}` with `user` and `model` roles. Its system prompt goes in `system_instruction`.

| Conversation role | OpenAI Chat | OpenAI Responses | Anthropic | Gemini |
|---|---|---|---|---|
| System prompt | `role: "system"` | `role: "system"` input or request instructions | Separate `system` parameter | `system_instruction` |
| `user` | `role: "user"` | `role: "user"` | `role: "user"` | `role: "user"` |
| `assistant` | `role: "assistant"` | `role: "assistant"` | `role: "assistant"` | `role: "model"` |
| `tool` | `role: "tool"` with `tool_call_id` | `function_call_output` with `call_id` and `output` | `tool_result` blocks in a `user` message | `functionResponse` parts in a `user` message |

Responses uses `input_text` for user text and `output_text` for assistant text. The shared model stores both as `Text`.

`Msg` has `role`, `content`, and `raw` fields. Tool identifiers, names, and arguments belong to the tool parts in `content`. There is no `Msg.data` metadata dict. Provider-specific fields, including error information, belong in `raw` rather than extra message fields. This module doesn't interpret them.

`raw` retains the original provider message. The corresponding `fastllm` assistant-message converter prefers it to rebuilding a message from parts. Don't use a provider's raw message with a different provider. This is distinct from a general provider-name override, which `Msg` doesn't support.

The original API comparison used OpenAI `ChatCompletionRequest*Message` (`specs/openai.with-code-samples.yml:35022-35444`), Anthropic `InputMessage` and `RequestToolResultBlock` (`specs/anthropic.yml:11140-11159,12595-12652`), and Gemini `Content` (`specs/gemini.json:171-188`).

In [ ]:
#| export
class Msg(BasicRepr):
    "A normalized message; `raw` is the wire form it was parsed from, when a provider produced it."
    def __init__(self,
        role,   # 'user', 'assistant', or 'tool'
        content, # list of `Part`
        raw=None # The provider's message as received, replayed verbatim to the same provider
    ):
        store_attr()

    @property
    def text(self): return ''.join(p.text or '' for p in self.content if isinstance(p, Text))

    def __eq__(self, o): return type(o) is type(self) and self.__dict__ == o.__dict__
    def __hash__(self): return hash((self.role, len(self.content)))

    def _repr_markdown_(self):
        return f"""**Msg**

- role: `{self.role}`

<contents>

{'\n\n'.join(p._repr_markdown_() for p in self.content)}

</contents>"""

In [ ]:
Msg('user', content=[Text('Hello world!', raw={'long':"10"*150})]*3)

**Msg**

- role: `user`

<contents>

**Text** (`text`)

Hello world!

::: details

- raw: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`
- citations: `None`

:::

**Text** (`text`)

Hello world!

::: details

- raw: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`
- citations: `None`

:::

**Text** (`text`)

Hello world!

::: details

- raw: `{'long': '1010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010101010...'}`
- citations: `None`

:::

</contents>

`msg2dict` writes the role, content, and raw provider message to a dict. Each part includes its registered `type` and public instance attributes. Underscore-prefixed attributes don't enter the dict.

`dict2msg` reconstructs the parts through `mk_part`. The example retains the provider's thinking signature through JSON serialization. Attribute values must themselves support JSON serialization. These helpers don't recursively convert arbitrary objects inside a part.

In [ ]:
#| export
def msg2dict(m):
    "`m` as a JSON-ready dict, each part tagged with its wire `type`"
    return dict(role=m.role, content=[dict(type=p.type, **{k:v for k,v in p.__dict__.items() if k[0] != '_'}) for p in m.content], raw=m.raw)

def dict2msg(d):
    "The `Msg` that `msg2dict` produced `d` from"
    return Msg(d['role'], [mk_part(**p) for p in d['content']], d.get('raw'))

In [ ]:
m = Msg('assistant', [Thinking('hmm', raw={'type':'thinking', 'signature':'sig'}), Text('Hi')], raw={'role':'assistant'})
d = msg2dict(m)
test_eq(dict2msg(json.loads(json.dumps(d))), m)
d

{'role': 'assistant',
 'content': [{'type': <PartType.thinking: 'thinking'>,
   'raw': {'type': 'thinking', 'signature': 'sig'},
   'cache_control': None,
   'text': 'hmm',
   'showthink': False},
  {'type': <PartType.text: 'text'>,
   'raw': None,
   'cache_control': None,
   'text': 'Hi',
   'citations': None}],
 'raw': {'role': 'assistant'}}

## Tool parts

`ToolUse` records a requested function call. `ToolResult` records its output. Both have `id`, `name`, `arguments`, `server`, and `text` fields. `arguments` defaults to an empty dict. `server=True` identifies a tool the provider ran itself.

Assistant messages can mix tool calls with text. Client tool results normally follow in a `tool` message. Provider responses can also include results from server tools. The formatted text representation combines each call with its result.

| API | Call object | Arguments | Streaming |
|---|---|---|---|
| [OpenAI Responses](https://developers.openai.com/api/reference/resources/responses/methods/create) | `{type: "function_call", call_id, name, arguments}` | JSON string in `arguments` | `response.function_call_arguments.delta` events |
| [OpenAI Chat](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create) | `{id, type: "function", function: {name, arguments}}` | JSON string in `function.arguments` | `tool_calls[i].function.arguments` deltas |
| [Anthropic](https://docs.anthropic.com/en/api/messages) | `{type: "tool_use", id, name, input}` | Parsed object in `input` | `input_json_delta` events |
| [Gemini](https://ai.google.dev/api/generate-content) | `functionCall: {id, name, args}` | Parsed object in `args` | Complete argument objects rather than JSON-text fragments |

`fastllm` converts these fields to `ToolUse.id`, `ToolUse.name`, and a parsed `ToolUse.arguments` dict. Responses supplies `call_id` for the ID. Chat supplies the function name from `function.name`. Anthropic and Gemini use `name` directly.

The original API comparison used OpenAI Responses `FunctionToolCall` (`specs/openai.with-code-samples.yml:44347-44389`), OpenAI Chat `ChatCompletionMessageToolCall` (`specs/openai.with-code-samples.yml:34869-34894`), Anthropic `ResponseToolUseBlock` (`specs/anthropic.yml:13563-13588`), and Gemini `FunctionCall` (`specs/gemini.json:273-293`).

In [ ]:
#| export
class _ToolPart(Part):
    "Shared shape of a tool call and its result."
    def __init__(self, id=None, name=None, arguments=None, server=False, text=None, **kw):
        if arguments is None: arguments = {}
        super().__init__(**kw)
        store_attr('id,name,arguments,server,text')

class ToolUse  (_ToolPart, tag=PartType.tool_use   ): "A tool invocation; `server` marks one the provider ran itself."
class ToolResult(_ToolPart, tag=PartType.tool_result): "A tool call's result, `text` holding the output."

In [ ]:
#| export
@patch
def _repr_markdown_(self:ToolUse):
    return f"""🔧 **{self.name}**(`{self.arguments}`)

::: details

- id: `{self.id}`
- server: `{self.server}`
- raw: `{_trunc_strs(self.raw)}`

:::"""

Use `display_list` to display parts or messages together as Markdown in a notebook.

In [ ]:
#| export
def display_list(l): 
    from IPython.display import Markdown, display
    display(Markdown('\n\n'.join(o._repr_markdown_() for o in l)))

In [ ]:
ToolUse(id='oxwvx1fm', name='simple_add', arguments={'b': 547982745, 'a': 5478954793}, raw={'thoughtSignature': 'EscDCsQDAQw51scPHdv+D5BX7JWdLzz3Bv8tsKFRuAJe2UkTFZ+NZKzNsLtmQBiia+/r4HJEUptq1zQB0q9HToX0qzCUqyNAbDLY76KxMeW9jpsnUvh6ZjPM5sDD7fAafF7cjdApNMsihPqIZBAZjAlFPcp1c/50MObH5f1q7hO7fgDS4iSJ3Q3FfbAYWnJ4nlA2peVMu/6WFcKZh1wcZCIuN6iFCj6nhH+6RKkaFRaM0b6XCmpti6qldSeZx+qtHmo+lzr1tct4Gz/CITDI7gRJ3qfLYV2u45jOhKzdd1t6gQ39XLJ93j0xd0AwpzcdZLbHWqwWJCQ43nNzhJ7IQTAWOSyPgKDnlAMHq2PTEoXBYkMBApCZ1x+HncBzt77kQrTTe7sWGVmD5boVnYAIFPFGXOULP5tDZ+nog+Fg8NV10vaFKlHVf+VDzFnVWxT259LN12ykGtBilfpTXiKCV12RAZwhuL7vXXHrsBGg5HNVImcXqgMvwf/rtQlJeop+9bEcAiU48hMFMzumOrCmmHD3HgxpYLW7T3vtDmbNdKCDqVtIwO4Rp5HE6GudRWmq8iC2UnyQglUXoXVnxIZW7eYYDsGAYrYgZ1A='})

🔧 **simple_add**(`{'b': 547982745, 'a': 5478954793}`)

::: details

- id: `oxwvx1fm`
- server: `False`
- raw: `{'thoughtSignature': 'EscDCsQDAQw51scPHdv+D5BX7JWdLzz3Bv8tsKFRuAJe2UkTFZ+NZKzNsLtmQBiia+/r4HJEUptq1zQB0q9HToX0qzCUqyNAbDLY...'}`

:::

## Completion

`Completion` contains an assistant message plus response metadata. `fastllm.types.mk_completion` constructs it from a non-streaming API response. It records the requested `model`, `api_name`, and `vendor_name`. `raw` holds the complete response dict.

The provider normalizers read different response fields:

| Value | OpenAI Responses | OpenAI Chat | Anthropic | Gemini |
|---|---|---|---|---|
| Content | `output[]`, including message content and function calls | `choices[0].message` | `content[]` | `candidates[0].content.parts[]` |
| Finish status | `status` | `choices[0].finish_reason` | `stop_reason` | `candidates[0].finishReason` |
| Usage | `usage` | `usage` | `usage` | `usageMetadata` |
| Tool calls | `function_call` output items | `message.tool_calls[]` | `tool_use` blocks | `functionCall` parts |

Each normalizer constructs the appropriate parts in `message.content`. Gemini retains separate text parts. `Completion.tool_calls` selects the `ToolUse` parts from that content rather than storing a second copy.

`finish_reason` uses normalized values such as `stop`, `tool_calls`, `length`, and `content_filter`. For example, Responses `completed`, Anthropic `end_turn`, and Gemini `STOP` normally become `stop`. A pending client tool call changes a successful finish to `tool_calls`. Usage normalization belongs to the provider modules.

The constructor uses the model name supplied by the caller for all providers. Gemini's `modelVersion` remains in `raw`; it doesn't replace the requested model name.

In [ ]:
#| export
class Completion(BasicRepr):
    "Normalized completion response."
    def __init__(self, model, message, finish_reason=None, usage=None, api_name=None, vendor_name=None, raw=None):
        if raw is None: raw = {}
        store_attr()
    def __eq__(self, o): return type(o) is type(self) and self.__dict__ == o.__dict__
    def __hash__(self): return hash((self.model, self.finish_reason, self.api_name, self.vendor_name))

    @property
    def tool_calls(self):
        "The `ToolUse` parts of `message`: a call lives in the content, not beside it"
        return [p for p in self.message.content if isinstance(p, ToolUse)]

In [ ]:
comp = Completion('claude-sonnet-4-20250514', Msg('assistant', [Text('The sum is 8. '), ToolUse(id='t1', name='add', arguments={'a':3,'b':5})]), finish_reason='tool_use')
test_eq(len(comp.tool_calls), 1)
comp.tool_calls[0]

🔧 **add**(`{'a': 3, 'b': 5}`)

::: details

- id: `t1`
- server: `False`
- raw: `None`

:::

## Message utilities

`mk_tool_res_msg` pairs calls and results by position. It returns one `tool` message containing a `ToolResult` per pair. Each result copies the call's ID, name, arguments, and server flag. Results can be strings or lists of media parts.

In [ ]:
#| export
def mk_tool_res_msg(tool_calls:list[ToolUse], results:list[str|list]):
    'A util to prepare parallel tool call with str or media list results'
    parts = [ToolResult(id=tc.id, name=tc.name, arguments=tc.arguments, server=tc.server, text=res)
        for tc,res in zip(tool_calls, results)]
    return Msg(role="tool", content=parts)

`sys_text` returns a string unchanged or reads a part's `text`. It also accepts `None`. `part_txt` reads `text` from a `Part` and returns other inputs unchanged.

In [ ]:
#| export
def sys_text(system):
    "Extract text from system (str or Part)."
    if system is None: return None
    return system if isinstance(system, str) else system.text

def part_txt(p): return p.text if isinstance(p,Part) else p

Provider converters need a MIME type for media. `data_url` extracts the MIME type and encoded body from a base64 data URL. `url_mime` first checks the URL extension. For an unknown extension, it fetches the resource and examines its `Content-Type` header. A missing or generic header triggers byte detection with `detect_mime`.

`_fetch_url_partial` imports `httpx` when you fetch a URL. The rest of this module doesn't require that import. `MediaUrl` stores a URL with an explicit or inferred MIME type. Constructing it without `mime` can therefore make a network request. It retains the URL rather than storing downloaded content.


In [ ]:
#| export
@flexicache(time_policy(24*3600))
def _fetch_url_partial(url, nbytes=512):
    "Fetch remote media, returning `(content_type_header, bytes)`; bytes optionally only the first `nbytes`."
    import httpx  # deliberately lazy: keeps aidialog's deps to fastcore alone (may re-base on fastcore.net later)
    try:
        with httpx.stream('GET', url, headers={'Range': f'bytes=0-{nbytes-1}'}, follow_redirects=True) as r:
            if r.status_code not in (200, 206): return None, None
            return r.headers.get('content-type'), r.read()
    except (httpx.HTTPError, httpx.InvalidURL): return None, None

In [ ]:
#| export
# chkstyle: skip
_ext_mime = {
    '.jpg':'image/jpeg', '.jpeg':'image/jpeg', '.png':'image/png', '.gif':'image/gif', '.webp':'image/webp', '.svg':'image/svg+xml',
    '.pdf':'application/pdf',
    '.mp3':'audio/mpeg', '.wav':'audio/wav', '.ogg':'audio/ogg', '.flac':'audio/flac', '.m4a':'audio/mp4',
    '.mp4':'video/mp4', '.mov':'video/quicktime', '.webm':'video/webm',
}

def data_url(url):
    "Parse data:mime;base64,data URL into (mime, b64_data), or None."
    if not isinstance(url, str) or not url.startswith('data:') or ',' not in url: return None
    header, body = url.split(',', 1)
    if ';base64' not in header or not body: return None
    return header[5:].split(';',1)[0].strip() or 'application/octet-stream', body

def url_mime(url, default='application/octet-stream'):
    "Guess mime from URL extension, then the server's Content-Type, then sniffed bytes; never None."
    if "youtube.com" in url or "youtu.be" in url: return "video/mp4"
    ext = '.' + url.rsplit('.', 1)[-1].split('?')[0].lower() if '.' in url.split('?')[0].split('/')[-1] else ''
    if (mime:=_ext_mime.get(ext)) is None:
        ctype, data = _fetch_url_partial(url)
        mime = (ctype or '').split(';')[0].strip().lower()
        if not mime or mime in ('application/octet-stream', 'text/plain'): mime = detect_mime(data)  # generic headers say nothing
    return ifnone(mime, default)

`url_mime` returns the default MIME type when neither the URL nor its content identifies a format. The default is `application/octet-stream`.

In [ ]:
test_eq(url_mime('https://example.com/badge.svg'), 'image/svg+xml')
test_eq(url_mime('nonsense.xyz123'), 'application/octet-stream')

These tests replace the fetch with local responses. A specific `Content-Type` takes priority after stripping parameters such as `charset`. For a generic header, recognizable PNG bytes determine the MIME type. Plain text with a generic header falls back to the default.

In [ ]:
_real = _fetch_url_partial
def _fetch_url_partial(url, nbytes=512): return 'image/svg+xml; charset=utf-8', b'<svg xmlns="http://www.w3.org/2000/svg"/>'
test_eq(url_mime('https://example.com/chart'), 'image/svg+xml')
def _fetch_url_partial(url, nbytes=512): return 'application/octet-stream', b'\x89PNG\r\n\x1a\n' + b'\0'*16
test_eq(url_mime('https://example.com/pic'), 'image/png')
def _fetch_url_partial(url, nbytes=512): return 'text/plain', b'some text'
test_eq(url_mime('https://example.com/thing'), 'application/octet-stream')
_fetch_url_partial = _real

In [ ]:
#| export
class MediaUrl(BasicRepr):
    "Direct URL media reference"
    def __init__(self, url, mime=None): self.url, self.mime = url, ifnone(mime, url_mime(url))

`mk_content` converts a string to `Text`. For bytes, it detects the MIME type and creates a base64 data URL. Unrecognized bytes raise `ValueError`. A `MediaUrl` becomes a media part that retains the URL.

The MIME prefix selects `InputImage`, `InputAudio`, or `InputVideo`. Other MIME types use `InputFile`. Message builders use this conversion for lists of mixed content.

In [ ]:
#| export
def _mime2part_cls(mime):
    "The `Media` subclass for MIME string `mime`"
    if mime.startswith('image/'): return InputImage
    if mime.startswith('audio/'): return InputAudio
    if mime.startswith('video/'): return InputVideo
    return InputFile

def _bytes2content(data):
    "Convert bytes to fastllm canonical content"
    mtype = detect_mime(data)
    if not mtype: raise ValueError(f'Data must be a supported file type, got {data[:10]}')
    encoded = base64.b64encode(data).decode("utf-8")
    return _mime2part_cls(mtype)(f'data:{mtype};base64,{encoded}', mime=mtype)

def _url2content(o):
    "Convert MediaUrl to fastllm canonical content"
    mime = o.mime or url_mime(o.url)
    return _mime2part_cls(mime)(o.url, mime=mime)

An existing `Part` passes through `mk_content` unchanged. Other unrecognized objects also pass through.

In [ ]:
#| export
def mk_content(o):
    "Convert a content value (str, bytes, `MediaUrl`, or already a `Part`) to a canonical `Part`"
    if isinstance(o, str):        return Text(o)
    elif isinstance(o, bytes):    return _bytes2content(o)
    elif isinstance(o, MediaUrl): return _url2content(o)
    return o

## The formatted text form

The editable reply format combines prose with fenced JSON. A `json {.tool}` block records a call and its result. A `json {.usage}` block records token usage. Tool blocks contain `id`, `name`, `args`, and `result`, with optional `server` and `md` flags. The `error` field is reserved.

Earlier `fastllm` releases used HTML `<details>` envelopes. The fixture below uses that older format. `conv_tools` converts it to fenced JSON. Running the conversion again leaves the result unchanged.

In [ ]:
fmt_outp = '''
I'll solve this step-by-step, using parallel calls where possible.

<details class='tool-usage-details' markdown='1'>

```json
{
  "id": "toolu_01KjnQH2Nsz2viQ7XYpLW3Ta",
  "call": { "function": "simple_add", "arguments": { "a": 10, "b": 5 } },
  "result": "15",
  "server": false
}
```

</details>

<details class='tool-usage-details' markdown='1'>

```json
{
  "id": "toolu_01Koi2EZrGZsBbnQ13wuuvzY",
  "call": { "function": "simple_add", "arguments": { "a": 2, "b": 1 } },
  "result": "3",
  "server": false
}
```

</details>

Now I need to multiply 15 * 3 before I can do the final division:

<details class='tool-usage-details' markdown='1'>

```json
{
  "id": "toolu_0141NRaWUjmGtwxZjWkyiq6C",
  "call": { "function": "multiply", "arguments": { "a": 15, "b": 3 } },
  "result": "45",
  "server": false
}
```

</details>

<details class='token-usage-details' markdown='1'><summary>Cache hit: 81.8% | Tokens: total=23,276 input=23,158 (+18,910 cached, 0 new) output=118 (reasoning 23)</summary>

`Usage(prompt_tokens=3, completion_tokens=10, total_tokens=13, raw={'input_tokens': 3, 'cache_creation_input_tokens': 2079, 'cache_read_input_tokens': 2070, 'cache_creation': {'ephemeral_5m_input_tokens': 2079, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 10, 'service_tier': 'standard', 'inference_geo': 'global'})`

</details>
'''

`parse_tools` reads valid tool blocks and returns the surrounding text with each parsed dict. Malformed JSON remains text. `strip_tools` removes tool and usage blocks by default. Its flags let you keep either kind.

`conv_tools` recognizes the released `<details markdown='1'>` envelopes and the unreleased `::: {.details}` spelling. Its legacy patterns are fixed to those formats.

In [ ]:
#| export
tool_info = 'json {.tool}'     # fence info string of a tool block: {id, name, args, result} (+server, md; `error` reserved)
usage_info = 'json {.usage}'   # fence info string of a usage block: UsageStats fields

def parse_tools(s):
    "Split `s` into `(text, data)` segments: `data` is a parsed `{.tool}` block dict, `None` for the final segment"
    res, pos = [], 0
    for info,body,start,end in fenced_blocks(s):
        if info != tool_info: continue
        try: d = json.loads(body)
        except Exception: continue
        res.append((s[pos:start], d))
        pos = end
    return res + [(s[pos:], None)]

def strip_tools(s, tools=True, usage=True):
    "Remove `{.tool}` (and `{.usage}`) blocks from `s`"
    out, pos = [], 0
    for info,body,start,end in fenced_blocks(s):
        if not (tools and info == tool_info) and not (usage and info == usage_info): continue
        out.append(s[pos:start])
        pos = end
    out.append(s[pos:])
    return ''.join(out)

think_start,think_end = '<!--think_start-->','<!--think_end-->'
re_think = re.compile(rf'{re.escape(think_start)}.*?{re.escape(think_end)}\n?', re.DOTALL)

# Frozen legacy envelope recognition, used only by `conv_tools`. These are the
# exact patterns fastllm shipped for the released `<details markdown='1'>`
# envelopes plus the never-released `::: {.details}` spelling.
_lg_tool_tag = "<details class='tool-usage-details' markdown='1'>"
_lg_token_tag = "<details class='token-usage-details' markdown='1'>"
_lg_tool_attrs, _lg_token_attrs = "{.details .tool-usage-details}", "{.details .token-usage-details}"
_lg_tools = re.compile(
    fr"^(?:{_lg_tool_tag}\n*(?:<summary>(?P<summ1>.*?)</summary>\n*)?\n*```json\n+(?P<json1>.*?)\n+```\n+</details>"
    fr"|(?P<fence>:{{3,}}) {re.escape(_lg_tool_attrs)}\n+(?:## (?P<summ2>.*?)\n+)?```json\n+(?P<json2>.*?)\n+```\n+(?P=fence)$)",
    flags=re.DOTALL|re.MULTILINE)
_lg_token = re.compile(
    fr"^(?:{re.escape(_lg_token_tag)}\n*<summary>(?P<tsumm1>.*?)</summary>\n*\n*`(?P<trepr1>.*?)`\n*\n*</details>"
    fr"|(?P<tfence>:{{3,}}) {re.escape(_lg_token_attrs)}\n+## (?P<tsumm2>.*?)\n+`(?P<trepr2>.*?)`\n+(?P=tfence)$)\n?",
    flags=re.DOTALL|re.MULTILINE)

def conv_tools(s):
    "Convert legacy tool/usage envelopes in `s` (both historical spellings) to the fenced JSON wire format. Idempotent."
    def _tool(m):
        tj = m['json1'] if m['json1'] is not None else m['json2']
        try: d = json.loads(tj.strip())
        except Exception: return m[0]
        call = d.get('call') or {}
        res = dict(id=d.get('id'), name=call.get('function'), args=call.get('arguments') or {}, result=d.get('result'))
        if d.get('server'): res['server'] = True
        return fenced(dumps(res, indent=2, ensure_ascii=False), tool_info)
    def _tok(m):
        summ = m['tsumm1'] if m['tsumm1'] is not None else m['tsumm2']
        det = m['trepr1'] if m['trepr1'] is not None else m['trepr2']
        return fenced(dumps(dict(summary=summ, detail=det), ensure_ascii=False), usage_info)
    return _lg_token.sub(_tok, _lg_tools.sub(_tool, s))

In [ ]:
wire_outp = conv_tools(fmt_outp)
test_eq(conv_tools(wire_outp), wire_outp)
assert 'json {.tool}' in wire_outp and 'details' not in wire_outp
Markdown(wire_outp)


I'll solve this step-by-step, using parallel calls where possible.

```json {.tool}
{
  "id": "toolu_01KjnQH2Nsz2viQ7XYpLW3Ta",
  "name": "simple_add",
  "args": {
    "a": 10,
    "b": 5
  },
  "result": "15"
}
```

```json {.tool}
{
  "id": "toolu_01Koi2EZrGZsBbnQ13wuuvzY",
  "name": "simple_add",
  "args": {
    "a": 2,
    "b": 1
  },
  "result": "3"
}
```

Now I need to multiply 15 * 3 before I can do the final division:

```json {.tool}
{
  "id": "toolu_0141NRaWUjmGtwxZjWkyiq6C",
  "name": "multiply",
  "args": {
    "a": 15,
    "b": 3
  },
  "result": "45"
}
```

```json {.usage}
{"summary": "Cache hit: 81.8% | Tokens: total=23,276 input=23,158 (+18,910 cached, 0 new) output=118 (reasoning 23)", "detail": "Usage(prompt_tokens=3, completion_tokens=10, total_tokens=13, raw={'input_tokens': 3, 'cache_creation_input_tokens': 2079, 'cache_read_input_tokens': 2070, 'cache_creation': {'ephemeral_5m_input_tokens': 2079, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 10, 'service_tier': 'standard', 'inference_geo': 'global'})"}
```

Each `parse_tools` segment contains the text before a tool block and that block's dict. The final segment contains the remaining text and `None`. `fmt2hist` uses these segments to reconstruct messages.

In [ ]:
segs = parse_tools(wire_outp)
test_eq(len(segs), 4)
[(txt.strip()[:40], d and d['name']) for txt,d in segs]

[("I'll solve this step-by-step, using para", 'simple_add'),
 ('', 'simple_add'),
 ('Now I need to multiply 15 * 3 before I c', 'multiply'),
 ('```json {.usage}\n{"summary": "Cache hit:', None)]

### Result fences

In [ ]:
#| export
_fence_back = '`````'
_result_re = re.compile(f'\n{_fence_back}result\n(.*?)\n{_fence_back}\n', re.DOTALL)
fence_call_re = re.compile(f'^{_fence_back}(py|bash)\n(.*?)\n{_fence_back}$', re.DOTALL | re.MULTILINE)


In [ ]:
#| export
def extract_fence_call(text):
    "Return (lang, code) if text ends with terminated py/bash fence, else None"
    ms = list(fence_call_re.finditer(text))
    if not ms: return None
    m = ms[-1]
    if not text[m.end():].strip(): return m.group(1), m.group(2)

`extract_fence_call` recognizes `py` and `bash` calls between five-backtick fences. Both fences must occupy their own lines. The closing fence must end the text apart from whitespace. It returns `(language, code)`, or `None` if no final call matches.

In [ ]:
for bad in ['\n`````py\nprint(1)', '\n```py\nprint(1)\n```\n', 'some text `````py\nprint(1)\n`````\n', '\n`````python\nprint(1)\n`````\n']:
    test_eq(bool(fence_call_re.search(bad)), False)
test_eq(extract_fence_call('prose\n`````bash\nls -la\n`````\n'), ('bash', 'ls -la'))
test_eq(extract_fence_call('\n`````py\nprint(1)\n`````\nmore text'), None)
test_eq(extract_fence_call('hello world'), None)
test_eq(extract_fence_call('\n`````py\nx = 1 | 2\n`````\n'), ('py', 'x = 1 | 2'))
extract_fence_call('\n`````py\nprint(1)\n`````\n')

('py', 'print(1)')

In [ ]:
#| export
def mk_result_fence(output): return f"\n{_fence_back}result\n{output}\n{_fence_back}\n"

def _split_msg_on_fences(msg):
    "Split an assistant Msg on result fences, return list of Msgs"
    if msg.role != 'assistant': return [msg]
    if not _result_re.search(msg.text): return [msg]
    res, asst_parts, tool_parts = [], [], []
    for p in msg.content:
        if   isinstance(p, Thinking): asst_parts.append(p)
        elif isinstance(p, ToolUse):  tool_parts.append(p)
        elif parts := _result_re.split(p.text or ''):
            for i,o in enumerate(parts):
                if not o: continue
                if i % 2 == 0: res.append(Msg(role='assistant', content=asst_parts+[Text(o.strip())]))
                else:          res.append(Msg(role='user',      content=[Text(mk_result_fence(o))]))
    if tool_parts: res.append(Msg(role='assistant', content=tool_parts))
    return res

def split_fence_msgs(msgs):
    "Split all assistant msgs on result fences for wire protocol"
    res = []
    for m in msgs: res.extend(_split_msg_on_fences(m))
    return res

`_split_msg_on_fences` separates assistant text at five-backtick `result` fences. It returns other messages unchanged. `split_fence_msgs` applies this operation to a list of messages.

In [ ]:
msg = Msg(role='assistant', content=[Text('Hello world')])
test_eq(_split_msg_on_fences(msg), [msg])
usr = Msg(role='user', content=[Text('`````result\n2\n`````')])
test_eq(_split_msg_on_fences(usr), [usr])
msg


**Msg**

- role: `assistant`

<contents>

**Text** (`text`)

Hello world

::: details

- raw: `None`
- citations: `None`

:::

</contents>

The model's code stays in an assistant message. The result becomes a `user` message containing its result fence, as required for code-result replay. Any following assistant text starts another message.

In [ ]:
msg = Msg(role='assistant', content=[Text('Let me calculate.\n`````py\n1+1\n`````\n\n`````result\n2\n`````\n\nDone.')])
res = _split_msg_on_fences(msg)
test_eq([m.role for m in res], ['assistant', 'user', 'assistant'])
test_eq(['`````py\n1+1' in res[0].text, '`````result\n2\n`````' in res[1].text, 'Done.' in res[2].text], [True]*3)
res

[Msg(role='assistant', content=[Text(raw=None, cache_control=None, text='Let me calculate.\n`````py\n1+1\n`````', citations=None)], raw=None),
 Msg(role='user', content=[Text(raw=None, cache_control=None, text='\n`````result\n2\n`````\n', citations=None)], raw=None),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text='Done.', citations=None)], raw=None)]

The split retains preceding `Thinking` parts with assistant text. The example has one assistant segment before the result.

In [ ]:
code_txt = '`````py\nimport random\nprint(random.random())\n`````\n`````result\n42\n`````\n'
msg = Msg(role='assistant', content=[Thinking('The user wants an RNG function...'), Text(code_txt)])
res = split_fence_msgs([msg])
test_eq([[type(p).__name__ for p in m.content] for m in res], [['Thinking','Text'], ['Text']])
res

[Msg(role='assistant', content=[Thinking(raw=None, cache_control=None, text='The user wants an RNG function...', showthink=False), Text(raw=None, cache_control=None, text='`````py\nimport random\nprint(random.random())\n`````', citations=None)], raw=None),
 Msg(role='user', content=[Text(raw=None, cache_control=None, text='\n`````result\n42\n`````\n', citations=None)], raw=None)]

The split places all `ToolUse` parts in a final assistant message. Tool requests end the assistant turn before the caller supplies their results.

In [ ]:
msg = Msg(role='assistant', content=[Text(code_txt),
    ToolUse(id='4vy96hyd', name='python', arguments={'code': 'print(random.randint(1, 100))'})])
res = _split_msg_on_fences(msg)
test_eq([m.role for m in res], ['assistant', 'user', 'assistant'])
test_eq([m.content[0].type for m in res], ['text', 'text', 'tool_use'])
res[-1]

**Msg**

- role: `assistant`

<contents>

🔧 **python**(`{'code': 'print(random.randint(1, 100))'}`)

::: details

- id: `4vy96hyd`
- server: `False`
- raw: `None`

:::

</contents>

### fmt2hist

`tool_text` converts a tool's return value to text for storage and provider requests. Strings pass through unchanged. Dicts and ordinary lists become JSON, not Python reprs. `default=str` converts values that JSON can't encode on their own. Other objects use `str`.

A list containing only `Part` objects follows a different rule. `tool_text` joins their `ctext` values with newlines. Text remains text. Media becomes a tag such as `<media input_image image/png>`, matching clikernel's kernel-image convention. `fastllm` uses this policy in its chat layer and provider serializers.

A stored tool block doesn't retain media bytes or URLs. Provider converters send the media during the live turn. Reading the block later recovers the placeholder text, not the media.

Two possible lossless formats remain deferred. Embedding media in the block would enlarge stored replies with base64. Display truncation could also corrupt it. Alternatively, a `ref` attribute on each `<media>` tag could identify content in a separate store, as Solveit's image attachments do. That would require `_media_tag` to write the reference and `_extract_tool_parts` to resolve it. Neither behavior exists yet.

In [ ]:
#| export
def _media_tag(p):
    m = getattr(p, 'mime', None)
    return f"<media {p.type} {m}>" if m else f"<media {p.type}>"

def tool_text(
    res, # A tool function's return value
):
    "Convert a tool result to text, using compact media tags and JSON for structured values."
    if isinstance(res, str): return res
    if isinstance(res, list) and all(isinstance(o, Part) for o in res): return '\n'.join(o.ctext for o in res)
    if isinstance(res, (dict, list)): return dumps(res, ensure_ascii=False, default=str)
    return str(res)

In [ ]:
test_eq(tool_text('hi'), 'hi')
test_eq(tool_text({'a': 1, 'ok': True}), '{"a": 1, "ok": true}')
test_eq(tool_text([1, 'x']), '[1, "x"]')
test_eq(tool_text(42), '42')
tool_text({'path': Path('/tmp')})

'{"path": "/tmp"}'

In [ ]:
#| export
def _extract_tool_parts(d:dict):
    "Build (tool_use_part, tool_result_part) from a parsed `{.tool}` block"
    if not d or d.get('id') is None: return None
    tu = ToolUse   (id=d['id'], name=d['name'], arguments=d.get('args') or {}, server=d.get('server', False))
    text = tool_text(d.get('result'))
    tr = ToolResult(id=d['id'], name=d['name'], text=MdStr(text) if d.get('md') else text, server=d.get('server', False))
    return tu, tr

In [ ]:
#| export
def fmt2hist(outp:str)->list[Msg]:
    "Transform a formatted output string into fastllm canonical Msgs"
    if usage_info in outp: outp = strip_tools(outp, tools=False)
    if think_start in outp: outp = re_think.sub('', outp)
    if tool_info not in outp:
        msg = Msg(role='assistant', content=[Text(outp.strip() or '.')])
        return _split_msg_on_fences(msg)
    hist, asst_parts, tool_parts = [], [], []
    def flush():
        if tool_parts:
            hist.append(Msg(role='assistant', content=asst_parts.copy()))
            hist.append(Msg(role='tool',      content=tool_parts.copy()))
            asst_parts.clear()
            tool_parts.clear()
    for txt,d in parse_tools(outp.strip()):
        if txt and txt.strip():
            if tool_parts: flush()
            asst_parts.append(Text(txt.strip() or '.'))
        if d and (tp := _extract_tool_parts(d)):
            asst_parts.append(tp[0])
            tool_parts.append(tp[1])
    flush()
    if asst_parts: hist.append(Msg(role='assistant', content=asst_parts))
    if not hist: hist.append(Msg(role='assistant', content=[Text('.')]))
    result = []
    for msg in hist:
        if msg.role == 'assistant': result.extend(_split_msg_on_fences(msg))
        else: result.append(msg)
    if result[-1].role == 'tool': result.append(Msg(role='assistant', content=[Text('.')]))
    return result

`fmt2hist` reconstructs alternating assistant and tool messages from the formatted reply. It removes usage and marked thinking blocks. An empty reply becomes an assistant message containing `.`. A reply ending in tool results also gets a final `.` assistant message.

In [ ]:
h = fmt2hist(wire_outp)
test_eq([m.role for m in h], ['assistant','tool','assistant','tool','assistant'])
h[1]

**Msg**

- role: `tool`

<contents>

**ToolResult** (`tool_result`)

15

::: details

- raw: `None`
- id: `toolu_01KjnQH2Nsz2viQ7XYpLW3Ta`
- name: `simple_add`
- arguments: `{}`
- server: `False`

:::

**ToolResult** (`tool_result`)

3

::: details

- raw: `None`
- id: `toolu_01Koi2EZrGZsBbnQ13wuuvzY`
- name: `simple_add`
- arguments: `{}`
- server: `False`

:::

</contents>

Wrap a tool return value in `ToolResponse` when downstream code must retain its structure instead of immediately stringifying it. The wrapper stores `content` unchanged. This can contain text or structured results such as image blocks. `fastllm` unwraps it in its tool-result handling.

In [ ]:
#| export
class ToolResponse(BasicRepr):
    def __init__(self, content): store_attr()  # list of (text, result) pairs
    def __eq__(self, o): return type(o) is type(self) and self.__dict__ == o.__dict__
    def __hash__(self): return hash(str(self.content))

These string subclasses request special handling downstream:

- `StopResponse` ends the tool loop after that tool result.
- `FullResponse` opts out of display truncation.
- `MdStr` allows the renderer to treat a tool result as Markdown rather than fenced text.

A tool must opt in to Markdown rendering. Untrusted Markdown can mislead readers. The formatted tool block records this choice in its `md` flag.

`trunc_str` also exempts fastcore's `Safe` and `PrettyString` types. It checks their class names without importing them. Ordinary strings can use `𝍁...𝍁` markers to request the same exemption after serialization.


In [ ]:
#| export
class StopResponse(str): pass
class FullResponse(str): pass
class MdStr(str): pass

`trunc_str` limits ordinary display strings and marks truncation with ellipses and a `TRUNCATED` wrapper. With the default `skip=10`, it drops the first ten characters as well as the tail. It does not preserve both ends. `mx=None` disables truncation.

`Safe`, `PrettyString`, and `FullResponse` pass through unchanged. With the default `replace`, `𝍁...𝍁` protects its contents from truncation and removes the markers. Setting `replace=None` disables that marker exemption.

In [ ]:
#| export
def trunc_str(s, mx=2000, skip=10, replace="TRUNCATED"):
    "Shorten ordinary display strings using `mx` and mark truncation with `replace`."
    if mx is None or isinstance_str(s, ('FullResponse','Safe','PrettyString')): return s
    if not isinstance(s, str): s = str(s)
    s = type(s)(s.rstrip())
    if len(s)>2 and s[0]=='𝍁' and s[-1]=='𝍁':
        s = s[1:-1]
        if replace: return s
    if mx is None or len(s)<=mx: return s
    s = s[skip:mx-skip]
    ss = s.split(' ')
    if len(ss[-1])>150: ss[-1] = ss[-1][:5]
    s = ' '.join(ss)
    if skip: s = f"…{s}"
    s = f"{s}…"
    if replace: s = f"<{replace}>{s}</{replace}>"
    return s

In [ ]:
test_eq(trunc_str('𝍁xxxxxxxxxx𝍁', mx=5), 'xxxxxxxxxx')
test_eq(trunc_str(Safe('xxxxxxxxxx'), mx=5), 'xxxxxxxxxx')
test_eq(trunc_str(FullResponse('xxxxxxxxxx'), mx=5), 'xxxxxxxxxx')
test_eq(trunc_str('xxxxxxxxxx', mx=5, skip=0), '<TRUNCATED>xxxxx…</TRUNCATED>')
test_eq(trunc_str('xxxxxxxxxx', mx=5, skip=1), '<TRUNCATED>…xxx…</TRUNCATED>')
test_eq(trunc_str('xxxxxxxxxx', mx=None), 'xxxxxxxxxx')

In [ ]:
#| export
def _trunc_param(v, mx=40):
    "Truncate and escape param value for display"
    tp = trunc_str(str(v).replace('`', r'\`'), mx=mx, replace=None, skip=0)
    try: return dumps(tp, ensure_ascii=False)
    except Exception: return repr(tp).replace('\\\\', '\\')

def _tc_summary(tr):
    "Format tool call as a `func(params)→result` code span"
    params = ', '.join(f"{k}={_trunc_param(v)}" for k,v in tr.arguments.items())
    res = f"→{_trunc_param(tr.text)}" if tr.text else ''
    txt = f"{tr.name}({params}){res}"
    ticks = '`'*(max(map(len, re.findall('`+', txt)), default=0)+1)
    pad = ' ' if '`' in txt else ''
    return f"{ticks}{pad}{txt}{pad}{ticks}"

In [ ]:
#| export
def mk_tr_details(tr, mx=2000):
    "Create the `{.tool}` wire block for a tool call; `mx=None` disables truncation"
    args = {k:trunc_str(v, mx=None if mx is None else mx*5) if isinstance(v, str) else v for k,v in tr.arguments.items()}
    res = dict(id=tr.id, name=tr.name, args=args, result=trunc_str(tool_text(tr.text), mx=mx))
    if tr.server: res['server'] = True
    if isinstance(tr.text, MdStr): res['md'] = True
    return "\n\n" + fenced(dumps(res, indent=2, ensure_ascii=False), tool_info) + "\n\n"

Parts implement two rendering interfaces. `formatted` supplies fragments for a streaming display. Text returns its current fragment without stripping whitespace. Thinking and pending tool calls have their own compact displays.

`doc(showthink=False, mx=2000)` renders finished content. It strips text, optionally includes thinking, and writes completed tool calls as fenced JSON. `hist2fmt` calls these methods while assembling a reply. Each subclass owns its rendering behavior.

In [ ]:
#| export
@patch(as_prop=True)
def formatted(self:Part): return self.text or ''
@patch
def doc(self:Part, showthink=False, mx=2000): return (self.text or '').strip()

`Thinking.formatted` returns the brain glyph by default. Setting the part's `showthink` attribute exposes the text during streaming. `Thinking.doc` instead uses its `showthink` argument. It puts the text in a collapsed details block between `think_start` and `think_end` comments. `fmt2hist` removes that block when reconstructing history.

In [ ]:
#| export
@patch(as_prop=True)
def formatted(self:Thinking): return (self.text or '') if self.showthink else '🧠'
@patch
def doc(self:Thinking, showthink=False, mx=2000):
    if not (showthink and self.text): return ''
    return f'{think_start}\n::: details\n\n## Thinking\n\n{self.text.strip()}\n\n:::\n{think_end}'

A client call's streaming display is a pending hourglass row. A server call has no streaming fragment here. Its document form is a completed tool block.

In [ ]:
#| export
@patch(as_prop=True)
def formatted(self:ToolUse): return '' if self.server else f"\n- ⏳ {_tc_summary(self)} ⏳\n"
@patch
def doc(self:ToolUse, showthink=False, mx=2000):
    "Render a completed server call or a pending client-call row."
    if not self.server: return self.formatted.strip()
    return mk_tr_details(self.replace(text=self.text or 'Server tool call executed.'), mx=mx).strip()

`ToolResult.doc` writes a tool block when the result has an ID. It returns an empty string without one. The parser needs an ID to reconstruct the matching call and result.

In [ ]:
#| export
@patch(as_prop=True)
def formatted(self:ToolResult): return mk_tr_details(self)
@patch
def doc(self:ToolResult, showthink=False, mx=2000):
    "Render an identified result as a tool block. Omit results without an ID."
    return mk_tr_details(self, mx=mx).strip() if self.id else ''

`ctext` supplies content inside a tool result. Most parts use their `formatted` value. `Media.ctext` uses a `<media>` tag instead of exposing a URL or a potentially large data URL.

In [ ]:
#| export
@patch(as_prop=True)
def ctext(self:Part): return self.formatted

@patch(as_prop=True)
def ctext(self:Media): return _media_tag(self)

In [ ]:
test_eq(Text('caption').ctext, 'caption')
test_eq(Thinking('secret').ctext, '🧠')
test_eq(tool_text([Text('caption'), InputImage('data:image/png;base64,xxx', mime='image/png')]), 'caption\n<media input_image image/png>')
test_eq(tool_text([InputFile('https://x.co/a.pdf')]), '<media input_file>')

This result contains a caption and an image. `mk_tr_details` uses `tool_text` to retain the caption and replace the image with a MIME-tagged placeholder. Reading this stored block won't restore the image.

In [ ]:
mtr = ToolResult(id='t1', name='screenshot', text=[Text('The login page.'), InputImage('data:image/png;base64,iVBORw0K', mime='image/png')])
assert '<media input_image image/png>' in mk_tr_details(mtr)
Markdown(mk_tr_details(mtr))



```json {.tool}
{
  "id": "t1",
  "name": "screenshot",
  "args": {},
  "result": "The login page.\n<media input_image image/png>"
}
```



In [ ]:
tc = ToolUse(id='tc1', name='simple_add', arguments={'a':3,'b':5})
test_eq(tc.formatted, '\n- ⏳ `simple_add(a="3", b="5")` ⏳\n')
test_eq(ToolUse(id='s1', name='web_search', server=True).formatted, '')
test_eq(Thinking('deep thought').formatted, '🧠')
test_eq(Thinking('deep thought', showthink=True).formatted, 'deep thought')
test_eq(Text('hi').formatted, 'hi')
tc.formatted

'\n- ⏳ `simple_add(a="3", b="5")` ⏳\n'

Use `hist2fmt` to turn captured assistant and tool messages into an editable reply, such as a Solveit prompt output or an llmsurgery dialog. It retains text and combines each matching `ToolUse`/`ToolResult` pair into a `json {.tool}` block. Other message roles raise `ValueError`.

`mk_tr_details` truncates long results using `mx`, which defaults to 2000. It allows string arguments up to five times that limit. Pass `mx=None` to disable those limits for matched call/result pairs. Server calls without a separate result still use their document method's default limit. The text format also discards raw provider data and media. It isn't a lossless serialization of arbitrary messages.

In [ ]:
#| export
def hist2fmt(msgs:list[Msg], mx=2000, showthink=False)->str:
    "Render assistant and tool messages as one editable Markdown reply."
    tus, out = {}, []
    for m in msgs:
        if m.role == 'assistant':
            for p in m.content:
                if isinstance(p, ToolUse): tus[p.id] = p
                else: out.append(p.doc(showthink=showthink, mx=mx))
        elif m.role == 'tool':
            for p in m.content:
                if not isinstance(p, ToolResult): continue
                tu = tus.pop(p.id, None)                       # results don't carry the call's arguments
                out.append(p.replace(arguments=tu.arguments if tu else {}).doc(mx=mx))
        else: raise ValueError(f"hist2fmt renders assistant and tool messages only, got {m.role!r}")
    out += [p.doc() for p in tus.values()]                     # calls still awaiting a result
    return '\n\n'.join(o for o in out if o)

In [ ]:
Markdown(hist2fmt(fmt2hist(wire_outp)))

I'll solve this step-by-step, using parallel calls where possible.

```json {.tool}
{
  "id": "toolu_01KjnQH2Nsz2viQ7XYpLW3Ta",
  "name": "simple_add",
  "args": {
    "a": 10,
    "b": 5
  },
  "result": "15"
}
```

```json {.tool}
{
  "id": "toolu_01Koi2EZrGZsBbnQ13wuuvzY",
  "name": "simple_add",
  "args": {
    "a": 2,
    "b": 1
  },
  "result": "3"
}
```

Now I need to multiply 15 * 3 before I can do the final division:

```json {.tool}
{
  "id": "toolu_0141NRaWUjmGtwxZjWkyiq6C",
  "name": "multiply",
  "args": {
    "a": 15,
    "b": 3
  },
  "result": "45"
}
```

.

The messages in this example contain text and completed tool calls within the display limit. They survive rendering and parsing unchanged.

In [ ]:
h2 = fmt2hist(hist2fmt(h))
test_eq(h2, h)

A client call without a result remains a pending row in the reply. `fmt2hist` treats that row as ordinary text. It doesn't recreate an unmatched `ToolUse`.

In [ ]:
pend = Msg('assistant', [Text('Calling...'), ToolUse(id='p1', name='f', arguments={'x':1})])
s = hist2fmt([pend])
test_eq(s, 'Calling...\n\n- ⏳ `f(x="1")` ⏳')
test_eq(fmt2hist(s)[0].content[0].text, s)
s

'Calling...\n\n- ⏳ `f(x="1")` ⏳'

A server call represents work the provider has already done. It doesn't need a separate result message. `hist2fmt` writes its `text` as the result in a completed tool block. If the text is empty, it uses `Server tool call executed.`.

`fmt2hist` reconstructs a call and result pair with `server=True`. The stored conversation therefore includes the result rather than requesting another execution. An `MdStr` result sets the block's `md` flag and returns as `MdStr` after parsing.

In [ ]:
srv = Msg('assistant', [Text('Let me check.'), ToolUse(id='s1', name='web_search', arguments={'query': 'otters'}, server=True, text=MdStr('Otter: https://example.com/otter'))])
s = hist2fmt([srv])
h3 = fmt2hist(s)
test_eq([m.role for m in h3[:2]], ['assistant', 'tool'])
test_eq((h3[0].content[1].server, h3[1].content[0].server), (True, True))
test_eq((h3[1].content[0].text, type(h3[1].content[0].text)), ('Otter: https://example.com/otter', MdStr))
test_eq(hist2fmt(h3[:2]), s)
Markdown(s)

Let me check.

```json {.tool}
{
  "id": "s1",
  "name": "web_search",
  "args": {
    "query": "otters"
  },
  "result": "Otter: https://example.com/otter",
  "server": true,
  "md": true
}
```

`hist2fmt` omits thinking unless you pass `showthink=True`. `fmt2hist` removes the marked thinking block in either case. The reconstructed history doesn't contain the thinking.

In [ ]:
tm = Msg('assistant', [Thinking('hmm, sums'), Text('The answer is 8.')])
test_eq(hist2fmt([tm]), 'The answer is 8.')
test_eq('hmm, sums' in hist2fmt([tm], showthink=True), True)
test_eq(fmt2hist(hist2fmt([tm], showthink=True))[0].content[0].text, 'The answer is 8.')
Markdown(hist2fmt([tm], showthink=True))

<!--think_start-->
::: details

## Thinking

hmm, sums

:::
<!--think_end-->

The answer is 8.

Gemini code-execution results can have no ID. `ToolResult.doc` omits them because it can't associate them with a call. The example retains the assistant text but loses the ID-less result.

In [ ]:
gm = Msg('assistant', [Text('Ran it.'), ToolResult(text='42', raw={'codeExecutionResult': {}})])
test_eq(hist2fmt([gm]), 'Ran it.')
test_eq(fmt2hist(hist2fmt([gm])), [Msg('assistant', [Text('Ran it.')])])

## Building conversations

`mk_msg` accepts a string, a list of mixed content, a role/content dict, a `Msg`, or a `Completion`. An existing `Msg` passes through unchanged. A `Completion` supplies its `message`. `None` returns `None`.

Strings become `Text`. A dict supplies its own role and wraps its `content` in `Text`. A list converts each item with `mk_content`. To convert bytes or a `MediaUrl` into media, put it in that list. A bare value takes the `Text` path. The default role is `user`.

In [ ]:
#| export
def mk_msg(
    content,      # Content: str, bytes (image), list of mixed content, or dict w 'role' and 'content' fields
    role="user"    # Message role if content isn't already a dict/Message
):
    "Build a `Msg` from content, or unwrap an existing message or completion."
    if content is None: return None
    if isinstance(content, Msg): return content
    if isinstance(content, Completion): return content.message
    if isinstance(content, list) and len(content) == 1 and isinstance(content[0], str): parts = [Text(content[0])]
    elif isinstance(content, list): parts = [mk_content(o) for o in content]
    elif isinstance(content, dict): return Msg(role=content['role'], content=[Text(content['content'])])
    else: parts = [Text(content)]
    return Msg(role=role, content=parts)

In [ ]:
m = mk_msg('hey')
test_eq(m, mk_msg(['hey']))
m

**Msg**

- role: `user`

<contents>

**Text** (`text`)

hey

::: details

- raw: `None`
- citations: `None`

:::

</contents>

`mk_msgs` builds a conversation. It starts with the `user` role for content without an explicit role. After a user or tool message, the next inferred role is `assistant`. After any other role, it is `user`.

Strings containing tool or usage fences first pass through `fmt2hist`. Existing messages and role/content dicts retain their roles. Empty input returns an empty list.

In [ ]:
#| export
def mk_msgs(
    msgs    # List of messages (each: str, bytes, list, Msg, or Completion)
):
    "Create a list of fastllm canonical Msgs."
    if not msgs: return []
    if not isinstance(msgs, list): msgs = [msgs]
    msgs = L(msgs).map(lambda m: fmt2hist(m) if isinstance(m,str) and (tool_info in m or usage_info in m) else [m]).concat()
    res, role = [], 'user'
    for m in msgs:
        res.append(msg := mk_msg(m, role=role))
        role = 'assistant' if msg.role in ('user','tool') else 'user'
    return res

Here's a conversation made from plain strings:

In [ ]:
msgs = mk_msgs(['Hey!',"Hi there!","How are you?","I'm doing fine and you?"])
msgs

[Msg(role='user', content=[Text(raw=None, cache_control=None, text='Hey!', citations=None)], raw=None),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text='Hi there!', citations=None)], raw=None),
 Msg(role='user', content=[Text(raw=None, cache_control=None, text='How are you?', citations=None)], raw=None),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text="I'm doing fine and you?", citations=None)], raw=None)]

In [ ]:
msgs[-2]

**Msg**

- role: `user`

<contents>

**Text** (`text`)

How are you?

::: details

- raw: `None`
- citations: `None`

:::

</contents>

Explicit tool roles also work for parallel calls. Both results below keep their `tool` role. The following plain string becomes the assistant's reply.

In [ ]:
msgs = mk_msgs(['Tell me the weather in Paris and Rome', 'Assistant calls weather tool two times',
    {'role': 'tool', 'content': 'Weather in Paris is ...'}, {'role': 'tool', 'content': 'Weather in Rome is ...'},
    'Assistant returns weather', 'Thanks!'])
msgs

[Msg(role='user', content=[Text(raw=None, cache_control=None, text='Tell me the weather in Paris and Rome', citations=None)], raw=None),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text='Assistant calls weather tool two times', citations=None)], raw=None),
 Msg(role='tool', content=[Text(raw=None, cache_control=None, text='Weather in Paris is ...', citations=None)], raw=None),
 Msg(role='tool', content=[Text(raw=None, cache_control=None, text='Weather in Rome is ...', citations=None)], raw=None),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text='Assistant returns weather', citations=None)], raw=None),
 Msg(role='user', content=[Text(raw=None, cache_control=None, text='Thanks!', citations=None)], raw=None)]

In [ ]:
#| hide
test_eq([m.role for m in msgs],['user','assistant','tool','tool','assistant','user'])

You can pass a single prompt without a list. `mk_msgs` wraps non-list input and returns a list of `Msg` objects. You'll get a ~~LiteLLM~~ fastllm message history back.

In [ ]:
msgs = mk_msgs("Hey")
msgs

[Msg(role='user', content=[Text(raw=None, cache_control=None, text='Hey', citations=None)], raw=None)]

In [ ]:
#| hide
msgs = mk_msgs({'role':'tool','content':'fake tool result'})
msgs

[Msg(role='tool', content=[Text(raw=None, cache_control=None, text='fake tool result', citations=None)], raw=None)]

In [ ]:
msgs = mk_msgs(['Hey!',"Hi there!","How are you?","I'm fine, you?"])
msgs

[Msg(role='user', content=[Text(raw=None, cache_control=None, text='Hey!', citations=None)], raw=None),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text='Hi there!', citations=None)], raw=None),
 Msg(role='user', content=[Text(raw=None, cache_control=None, text='How are you?', citations=None)], raw=None),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text="I'm fine, you?", citations=None)], raw=None)]

For one message with multiple parts, use two lists. The outer list contains messages. The inner list contains that message's parts.

This is useful for a question with an image. The example below contains two text parts: a question and a URL string. Use a `MediaUrl` or `InputImage` instead of a plain URL string to send the image itself.

In [ ]:
msgs = mk_msgs([['Whats in this image?', 'https://example.com/puppy.jpg']])
test_eq(len(msgs), 1)
test_eq(len(msgs[0].content), 2)
msgs[0]

**Msg**

- role: `user`

<contents>

**Text** (`text`)

Whats in this image?

::: details

- raw: `None`
- citations: `None`

:::

**Text** (`text`)

https://example.com/puppy.jpg

::: details

- raw: `None`
- citations: `None`

:::

</contents>

In [ ]:
#| hide
#| eval: false
import nbdev
nbdev.nbdev_export()